In [13]:
# 📊 Mini Data Quality Audit

In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)

In [ ]:
## 📌 Dataset Description

The dataset contains customer transaction records, including attributes such as customer identifiers, transaction dates, and transaction amounts.

### 🧠 Possible Business Use Case
- Sales performance analysis  
- Customer behavior tracking  
- Revenue forecasting  
- Fraud or anomaly detection  

In [2]:
# Load dataset
df = pd.read_csv("week2_customer_transactions_messy.csv")

# Preview
df.head()

,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


In [ ]:
## 📌 Data Quality Assessment

In [3]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nData Types:\n", df.dtypes)

Shape: (11, 9)

Columns:
 Index(['transaction_id', 'customer_id', 'transaction_date', 'amount',
       'currency', 'payment_method', 'status', 'region', 'last_updated'],
      dtype='object')

Data Types:
 transaction_id       object
customer_id          object
transaction_date     object
amount              float64
currency             object
payment_method       object
status               object
region               object
last_updated         object
dtype: object


In [ ]:
### 🔹 Completeness
- Missing values detected across multiple columns  
- Reduces reliability of analysis  

In [4]:
# Missing values
missing_values = df.isnull().sum()

# Duplicate rows
duplicate_rows = df.duplicated().sum()

# Basic stats
summary_stats = df.describe(include='all')

missing_values, duplicate_rows, summary_stats

(transaction_id      0
 customer_id         1
 transaction_date    0
 amount              1
 currency            0
 payment_method      1
 status              0
 region              0
 last_updated        1
 dtype: int64,
 np.int64(1),
        transaction_id customer_id transaction_date          amount currency  \
 count              11          10               11       10.000000       11   
 unique             10           9                8             NaN        3   
 top             T0006        C105       2026-01-07             NaN      EUR   
 freq                2           2                2             NaN        9   
 mean              NaN         NaN              NaN   100056.457000      NaN   
 std               NaN         NaN              NaN   316207.939002      NaN   
 min               NaN         NaN              NaN      -35.000000      NaN   
 25%               NaN         NaN              NaN       19.990000      NaN   
 50%               NaN         NaN          

In [5]:
missing_values[missing_values > 0]

customer_id       1
amount            1
payment_method    1
last_updated      1
dtype: int64

In [ ]:
### 🔹 Uniqueness
- Duplicate records present in dataset  
- Can lead to double counting of transactions  

In [6]:
df[df.duplicated()]

,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15


In [ ]:
### 🔹 Validity
- Some transaction amounts are zero or negative  
- Violates expected business rules  

In [7]:
# Example: negative or zero transaction amounts
if 'Transaction_Amount' in df.columns:
    invalid_amounts = df[df['Transaction_Amount'] <= 0]
    invalid_amounts

In [ ]:
### 🔹 Consistency
- Inconsistent text formats (e.g., uppercase/lowercase differences)  
- Affects grouping and aggregation  

In [8]:
# Example: inconsistent text (case differences)
for col in df.select_dtypes(include='object').columns:
    print(f"\nUnique values in {col}:\n", df[col].unique()[:10])


Unique values in transaction_id:
 ['T0001' 'T0002' 'T0003' 'T0004' 'T0005' 'T0006' 'T0007' 'T0008' 'T0009'
 'T0010']

Unique values in customer_id:
 ['C100' 'C101' 'C102' nan 'C104' 'C105' 'C106' 'C107' 'C108' 'C109']

Unique values in transaction_date:
 ['2026-01-05' '2026/01/06' '06-01-2026' '2026-01-07' '2026-01-08'
 '2026-02-30' '2026-01-10' '2026-01-11']

Unique values in currency:
 ['EUR' 'USD' 'EURO']

Unique values in payment_method:
 ['card' 'CARD' 'bank_transfer' 'cash' nan]

Unique values in status:
 ['completed' 'pending' 'cancelled']

Unique values in region:
 ['DE' 'de' 'US' 'FR' 'NL']

Unique values in last_updated:
 ['2026-01-05' '2026-01-20' '2026-01-07' '2026-01-08' '2026-01-09'
 '2026-04-15' '2026-02-15' '2026-01-10' nan '2026-01-11']


In [ ]:
### 🔹 Integrity
- Missing or invalid Customer IDs  
- Breaks relationships between entities  

In [9]:
# Example: check if customer_id exists
if 'Customer_ID' in df.columns:
    df[df['Customer_ID'].isnull()]

In [ ]:
## 📌 KPI Summary

| KPI                  | Description                                  |
|----------------------|----------------------------------------------|
| Completeness Rate    | Percentage of non-missing values             |
| Duplication Rate     | Percentage of duplicate rows                 |
| Error Rate           | Percentage of invalid records                |

In [10]:
total_cells = df.size
missing_cells = df.isnull().sum().sum()

# 1. Completeness Rate
completeness_rate = 1 - (missing_cells / total_cells)

# 2. Duplication Rate
duplication_rate = df.duplicated().sum() / len(df)

# 3. Error Rate (example: invalid transaction amounts)
if 'Transaction_Amount' in df.columns:
    error_rate = (df['Transaction_Amount'] <= 0).sum() / len(df)
else:
    error_rate = np.nan

completeness_rate, duplication_rate, error_rate

(np.float64(0.9595959595959596), np.float64(0.09090909090909091), nan)

In [ ]:
### 🧠 Interpretation
- Lower completeness → more missing data issues  
- Higher duplication → redundant records  
- Higher error rate → poor data validity  

In [ ]:
## 📌 Validation Rules

1. Transaction amount must be greater than 0  
2. Customer ID must not be null  
3. Transaction date must be in valid datetime format  

In [11]:
validation_issues = pd.DataFrame()

# Rule 1: Transaction amount must be > 0
if 'Transaction_Amount' in df.columns:
    validation_issues['invalid_amount'] = df['Transaction_Amount'] <= 0

# Rule 2: Customer ID must not be null
if 'Customer_ID' in df.columns:
    validation_issues['missing_customer'] = df['Customer_ID'].isnull()

# Rule 3: Date must be valid format
if 'Transaction_Date' in df.columns:
    validation_issues['invalid_date'] = pd.to_datetime(
        df['Transaction_Date'], errors='coerce'
    ).isnull()

validation_issues.head()

""


In [ ]:
## 📌 Audit Summary

| Issue Type                  | Affected Rows | Severity | Recommended Action                     |
|----------------------------|--------------|----------|----------------------------------------|
| Missing Values             | High         | High     | Impute or remove missing values        |
| Duplicate Records          | Medium       | Medium   | Remove duplicates                      |
| Invalid Transaction Amount | Medium       | High     | Filter or correct invalid values       |
| Invalid Dates              | Low          | Medium   | Convert and fix date formats           |

In [12]:
audit_summary = pd.DataFrame({
    "Issue Type": [
        "Missing Values",
        "Duplicate Records",
        "Invalid Transaction Amount",
        "Invalid Dates"
    ],
    "Affected Rows": [
        missing_cells,
        df.duplicated().sum(),
        (df['Transaction_Amount'] <= 0).sum() if 'Transaction_Amount' in df.columns else 0,
        validation_issues['invalid_date'].sum() if 'invalid_date' in validation_issues else 0
    ],
    "Severity": [
        "High",
        "Medium",
        "High",
        "Medium"
    ],
    "Recommended Action": [
        "Impute or remove missing values",
        "Remove duplicates",
        "Filter or correct invalid values",
        "Convert and fix date formats"
    ]
})

audit_summary

,Issue Type,Affected Rows,Severity,Recommended Action
0,Missing Values,4,High,Impute or remove missing values
1,Duplicate Records,1,Medium,Remove duplicates
2,Invalid Transaction Amount,0,High,Filter or correct invalid values
3,Invalid Dates,0,Medium,Convert and fix date formats


In [ ]:
## ✅ Final Summary

This audit identified key data quality issues across completeness, uniqueness, validity, consistency, and integrity.

KPIs provide insight into dataset health, while validation rules and recommended cleaning steps define the next actions to improve data quality.